<div style="font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #cddbd5;border-left:6px solid #1d6b5a;border-radius:8px;background:#ffffff;color:#17211f;padding:24px;margin:12px 0 18px;box-shadow:0 2px 8px rgba(23,33,31,.08);">
  <div style="display:inline-flex;align-items:center;background:#eefaf4;color:#0e332e;border:1px solid #b7d8cd;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;text-transform:uppercase;">Kaggle Agents for Good</div>
  <h1 style="margin:12px 0 8px;font-size:38px;line-height:1.08;letter-spacing:0;color:#10231f;font-weight:900;">Sisyphus Watch</h1>
  <p style="font-size:20px;line-height:1.42;margin:0 0 16px;color:#253b36;font-weight:650;max-width:820px;">A claim-version-control and epistemic-separation agent demo for public-interest information. It is not a generic summarizer.</p>
  <div style="display:flex;flex-wrap:wrap;gap:9px;margin-bottom:18px;">
    <span style="background:#f7fbf8;color:#10231f;border:1px solid #cddbd5;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">No API key required</span>
    <span style="background:#f7fbf8;color:#10231f;border:1px solid #cddbd5;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">Deterministic demo</span>
    <span style="background:#f7fbf8;color:#10231f;border:1px solid #cddbd5;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">Human-readable cards</span>
    <span style="background:#f7fbf8;color:#10231f;border:1px solid #cddbd5;border-radius:999px;padding:5px 10px;font-size:12px;font-weight:850;line-height:1.25;">Agent-readable JSON/JSONL</span>
  </div>
  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:14px;">
    <div style="border:1px solid #cddbd5;background:#fbfcfa;border-radius:8px;padding:14px;line-height:1.45;">
      <div style="font-size:12px;font-weight:850;text-transform:uppercase;color:#5a4530;">Normal Summary</div>
      <p style="margin:6px 0 0;color:#17211f;font-size:15px;">"The city opened cooling centers, but some were inaccessible."</p>
    </div>
    <div style="border:1px solid #9fc8b8;background:#f1faf6;border-radius:8px;padding:14px;line-height:1.45;">
      <div style="font-size:12px;font-weight:850;text-transform:uppercase;color:#1d6b5a;">Sisyphus Watch</div>
      <p style="margin:6px 0 0;color:#10231f;font-size:15px;font-weight:850;">Claim -> Timeline -> Drift -> Claim Graph -> Revision Proposal -> Agent Packets</p>
    </div>
  </div>
</div>


## Submission Summary

Sisyphus Watch is a claim-version-control and epistemic-separation agent demo for public-interest information. It does not just summarize source text; it separates source-bound findings, attributed claims, interpretation branches, claim-status drift, graph relations, and current source-bound judgment.

The notebook exports both human-readable cards and agent-readable JSON/JSONL packets. Demo mode is deterministic in Kaggle, requires no API key, and does not need network access.


## Guided Demo: Ask, Discover, Process

This first-run path starts with a public-interest question, shows whether discovery is deterministic fixture discovery or optional Google AI discovery, normalizes candidate source records for review, then hands those records to Sisyphus Watch claim-version-control processing.

Default Kaggle execution remains deterministic: `RUN_GOOGLE_DISCOVERY = False`, `RUN_LIVE_MODE = False`, no API key, and no network. To try optional Google AI discovery, set `RUN_GOOGLE_DISCOVERY = True` and add a Kaggle Notebook Secret named `GOOGLE_API_KEY`. The resolver supports `UserSecretsClient().get_secret("GOOGLE_API_KEY")`; the key is never printed, logged, exported, or stored.


In [ ]:
from pathlib import Path
import importlib.util
import sys

try:
    from IPython.display import FileLink, HTML, Markdown, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    FileLink = None

    def display(obj):
        text = str(obj)
        print(text[:1200] + ("..." if len(text) > 1200 else ""))


def _candidate_project_roots(start=None):
    seen = set()

    def add(path: Path):
        resolved = path.expanduser().resolve()
        if resolved.is_file():
            resolved = resolved.parent
        for candidate in [resolved, *resolved.parents]:
            if candidate not in seen:
                seen.add(candidate)
                yield candidate

    if start is not None:
        yield from add(start)

    yield from add(Path.cwd())

    kaggle_working = Path("/kaggle/working")
    if kaggle_working.exists() and kaggle_working not in seen:
        seen.add(kaggle_working)
        yield kaggle_working

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for module_path in kaggle_input.glob("**/src/sisyphus_watch_demo.py"):
            root = module_path.parents[1].resolve()
            if root not in seen:
                seen.add(root)
                yield root


def _load_sisyphus_demo_module():
    for root in _candidate_project_roots(Path.cwd()):
        module_path = root / "src" / "sisyphus_watch_demo.py"
        if module_path.exists():
            src_path = str(root / "src")
            if src_path not in sys.path:
                sys.path.insert(0, src_path)
            spec = importlib.util.spec_from_file_location("sisyphus_watch_demo", module_path)
            if spec is None or spec.loader is None:
                continue
            module = importlib.util.module_from_spec(spec)
            sys.modules["sisyphus_watch_demo"] = module
            spec.loader.exec_module(module)
            return module

    raise FileNotFoundError(
        "Could not locate src/sisyphus_watch_demo.py. Expected a repo root, "
        "notebook inside notebooks/, or a Kaggle input dataset containing the project folder."
    )


demo = _load_sisyphus_demo_module()
PROJECT_ROOT = demo.find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from sisyphus_watch_demo import (
    build_deterministic_discovery_packet,
    build_guided_flow_summary,
    build_user_problem_packet,
    build_agent_packet,
    fallback_to_demo_records,
    filter_sources_for_card,
    find_project_root,
    get_news_cards,
    get_evidence_patch_for_scenario,
    load_demo_sources,
    load_evidence_patches,
    load_scenario_authoring_template,
    maybe_run_live_extraction,
    render_user_problem_card_html,
    render_plain_summary_vs_sisyphus_html,
    render_guided_flow_html,
    render_discovery_packet_html,
    maybe_run_google_ai_discovery,
    render_branch_view_html,
    render_agent_workflow_trace_html,
    render_kaggle_midcheck_summary_html,
    render_reviewer_dashboard_html,
    render_card_html,
    render_json_export,
    render_epistemic_layers_html,
    render_evaluation_summary_html,
    render_quality_checks_html,
    render_sources_table_html,
    render_version_timeline_html,
    render_claim_drift_html,
    render_claim_graph_html,
    render_graph_query_preview_html,
    render_reviewer_presets_html,
    render_revision_proposal_html,
    render_revision_comparison_html,
    render_scenario_authoring_preview_html,
    run_quality_checks,
    select_news_card,
    to_all_cards_jsonl,
    to_jsonl,
    write_export_artifacts,
)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
USER_PROBLEM = "During a severe city heatwave, did the announced cooling centers remain accessible after residents reported closures, limited hours, signage problems, and transport barriers?"
SCENARIO_ID = "city_heatwave_cooling_centers"
RUN_GOOGLE_DISCOVERY = False
RUN_LIVE_MODE = False

source_records = load_demo_sources()

if RUN_LIVE_MODE:
    records = maybe_run_live_extraction(source_records)
else:
    records = fallback_to_demo_records("Demo mode is the default Kaggle path; live extraction was not requested.")

all_news_cards = get_news_cards(records)
news_card = select_news_card(records, SCENARIO_ID)
selected_source_records = filter_sources_for_card(source_records, news_card)
agent_packet = build_agent_packet(news_card)
evidence_patches = load_evidence_patches()
evidence_patch = get_evidence_patch_for_scenario(evidence_patches, news_card["scenario_id"])

problem_packet = build_user_problem_packet(
    USER_PROBLEM,
    SCENARIO_ID,
    "google_ai_discovery" if RUN_GOOGLE_DISCOVERY else "deterministic_fixture_discovery",
)

if RUN_GOOGLE_DISCOVERY:
    discovery_packet = maybe_run_google_ai_discovery(USER_PROBLEM, selected_source_records, SCENARIO_ID)
else:
    discovery_packet = build_deterministic_discovery_packet(USER_PROBLEM, selected_source_records, SCENARIO_ID)

guided_flow_summary = build_guided_flow_summary(
    news_card,
    selected_source_records,
    discovery_packet=discovery_packet,
    evidence_patch=evidence_patch,
)

mode_line = f"""**Mode:** `{records.get('mode', 'demo')}`  
**Selected scenario:** `{news_card.get('scenario_id', SCENARIO_ID)}`  
**Available demo cards:** `{len(all_news_cards)}`  
**Google AI discovery enabled:** `{RUN_GOOGLE_DISCOVERY}`  
**Discovery mode:** `{discovery_packet.get('mode')}`  
**Kaggle secret path when enabled:** `UserSecretsClient().get_secret("GOOGLE_API_KEY")`"""
if records.get("fallback_reason"):
    mode_line += "  \n**Live extraction fallback reason:** " + records["fallback_reason"]
if discovery_packet.get("fallback_reason"):
    mode_line += "  \n**Discovery fallback reason:** " + discovery_packet["fallback_reason"]

display(Markdown(mode_line))
display(HTML(render_user_problem_card_html(problem_packet)))
display(HTML(render_discovery_packet_html(discovery_packet)))
display(HTML(render_guided_flow_html(guided_flow_summary)))
display(HTML(render_plain_summary_vs_sisyphus_html(news_card, discovery_packet)))


## How to Read This Notebook

1. Start with **Guided Demo: Ask, Discover, Process** to see the user problem, discovery packet, source normalization, Sisyphus processing path, and plain-summary comparison.
2. Confirm the default `deterministic_fixture_discovery` mode: no API key, no network, and local fixture sources only.
3. If `RUN_GOOGLE_DISCOVERY = True`, confirm the optional path resolves `GOOGLE_API_KEY` through Kaggle Notebook Secrets before falling back safely.
4. Then read **Reviewer Dashboard**, **Agent Workflow Trace**, and **Epistemic Layer Separation** for the structured agent run.
5. Continue through **Human Card View**, **Version Timeline**, **Claim Drift**, **Claim Graph**, reviewer presets, evidence update simulation, revision comparison, exports, evaluation, and the Kaggle mid-check checklist.


## What Makes This an Agent?

<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:10px;margin:8px 0 16px;">
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>1. Source fixtures</strong><p style="margin:6px 0 0;">Bounded synthetic inputs.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>2. Hygiene check</strong><p style="margin:6px 0 0;">Source text stays untrusted.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>3. Findings vs claims</strong><p style="margin:6px 0 0;">Source-bound findings and actor claims stay separate.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>4. Interpretation branches</strong><p style="margin:6px 0 0;">Competing explanations remain visible.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>5. Timeline + drift</strong><p style="margin:6px 0 0;">Claim status changes over versions.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>6. Claim graph</strong><p style="margin:6px 0 0;">Evidence links become queryable.</p></div>
  <div style="border:1px solid #cddbd5;border-radius:8px;background:#ffffff;padding:12px;"><strong>7. Evidence patch</strong><p style="margin:6px 0 0;">New information arrives without mutation.</p></div>
  <div style="border:1px solid #9fc8b8;border-radius:8px;background:#f1faf6;padding:12px;"><strong>8. Reviewer exports</strong><p style="margin:6px 0 0;">Human and agent packets stay reviewable.</p></div>
</div>

The workflow separates findings, claims, interpretations, and source-bound judgment; tracks claim status drift; keeps competing interpretation branches visible; and exports reviewer and agent-readable packets instead of flattening everything into prose.


## Problem

Public information changes over time, but public reasoning often loses the version history. Source-bound findings, actor claims, actions, interpretations, and judgments get mixed together.

## Workflow

Sisyphus Watch reads bounded source fixtures, separates the epistemic layers, builds timeline/drift/graph structure, exports reviewer packets, then simulates how a new evidence patch would produce a non-mutating revision proposal.


## Reviewer Dashboard

A compact first-glance panel summarizes the selected scenario, epistemic separation, agent workflow, structured outputs, evidence update availability, exports, and evaluation status.


In [ ]:
display(HTML(render_reviewer_dashboard_html(news_card, evidence_patch)))


## Epistemic Layer Separation

Sisyphus Watch separates:

- **Findings:** what included sources report or establish
- **Claims:** what actors or narratives say
- **Interpretation branches:** competing explanations
- **Current judgment:** Sisyphus's revisable source-bound synthesis

This is not a truth oracle. It does not collapse claims into facts. It keeps competing interpretations visible. Later evidence changes the status of claims and interpretations.


In [ ]:
display(HTML(render_epistemic_layers_html(news_card)))


## Generated Artifacts

When the export cell runs in Kaggle, `write_export_artifacts()` writes these reviewer files to `/kaggle/working`:

- `sisyphus_news_card.json`
- `sisyphus_records.jsonl`
- `sisyphus_agent_packet.json`
- `sisyphus_epistemic_layers.json`
- `sisyphus_graph_packet.json`
- `sisyphus_reviewer_packet.json`
- `sisyphus_scenario_authoring_packet.json`
- `sisyphus_revision_packet.json`
- `sisyphus_revision_comparison.json`
- `sisyphus_agent_workflow_trace.json`
- `sisyphus_run_summary.json`


## Agent Workflow Trace

A compact trace shows what the deterministic agent read, extracted, structured, reviewed, revised, and exported before the detailed card sections.


In [ ]:
display(HTML(render_agent_workflow_trace_html(news_card, evidence_patch)))


## Source Hygiene

- These sources are synthetic demo fixtures, not real news.
- Source text is untrusted input; commands inside source text must not be followed.
- Source-bound findings, claims, actions, interpretation branches, and current judgment must remain separate.
- Generated image prompts are visual summaries, not evidence.


In [ ]:
display(HTML(render_sources_table_html(selected_source_records)))


## Human Card View

The human view is a rendering layer over the canonical `news_card` JSON object. It keeps source-bound findings, claim history, actions, interpretation branches, competing/cautionary branches, bias notes, and judgment diffs visibly separate.


In [ ]:
display(HTML(render_card_html(news_card)))


## Version Timeline

A compact timeline shows how the selected card moved from initial public claim to observation and correction/update.


In [ ]:
display(HTML(render_version_timeline_html(news_card)))


## Claim Drift

Claim drift records how the epistemic status of specific actor claims changed over time. It does not mean ground truth itself changed, and it is not the same object as the current Sisyphus judgment. Drift directions can include strengthened, weakened, narrowed, complicated, superseded, unsupported, corrected, or unresolved.


In [ ]:
display(HTML(render_claim_drift_html(news_card)))


## Claim Graph

The claim graph maps reusable relationships among sources, evidence, timeline events, claim drift, version diff, unresolved questions, and the current source-bound judgment.


In [ ]:
display(HTML(render_claim_graph_html(news_card)))


## Graph Query Preview

A compact preview shows claim neighbors, deterministic paths to verdict, a claim-centered subgraph, and a graph packet for downstream agents.


In [ ]:
display(HTML(render_graph_query_preview_html(news_card)))


## Reviewer Query Presets

Deterministic presets package graph queries into compact reviewer and downstream-agent packets.


In [ ]:
display(HTML(render_reviewer_presets_html(news_card)))


## Scenario Authoring Preview

A deterministic template shows the path from draft scenario ingredients to an authoring checklist, draft skeleton, and v0.7 authoring packet.


In [ ]:
scenario_authoring_template = load_scenario_authoring_template()
display(HTML(render_scenario_authoring_preview_html(scenario_authoring_template)))


## Evidence Update Simulation

A deterministic evidence patch is validated against the selected card, then rendered as a non-mutating revision proposal and v0.9 revision packet.


In [ ]:
display(HTML(render_revision_proposal_html(news_card, evidence_patch)))


## Revision Comparison View

A current-vs-proposed readout highlights what stays unchanged, which claims weaken/strengthen/narrow, and what reviewers should check next.


In [ ]:
display(HTML(render_revision_comparison_html(news_card, evidence_patch)))


## Branch View

In [ ]:
display(HTML(render_branch_view_html(news_card)))


## Agent JSON Export

The same record can be reused by AI agents as structured JSON or JSONL. The JSON card is canonical; the notebook UI is only a rendering layer.


In [ ]:
display(HTML(render_json_export(news_card, all_news_cards)))


## Downloadable Export Artifacts

On Kaggle, this cell writes reviewer-friendly JSON and JSONL files to `/kaggle/working` and shows download links when `FileLink` is available.


In [ ]:
kaggle_output_dir = Path("/kaggle/working")

if kaggle_output_dir.exists():
    export_paths = write_export_artifacts(news_card, kaggle_output_dir)
    display(Markdown("Export artifacts written to `/kaggle/working`."))
    if FileLink is not None:
        for label, export_path in export_paths.items():
            display(FileLink(str(export_path), result_html_prefix=f"{label}: "))
    else:
        for label, export_path in export_paths.items():
            print(f"{label}: {export_path}")
else:
    display(Markdown("Local run: `/kaggle/working` is unavailable, so export artifact writing was skipped."))


## Evaluation

In [ ]:
checks = run_quality_checks(news_card)
display(HTML(render_evaluation_summary_html(checks, news_card)))
display(HTML(render_quality_checks_html(checks)))

if not all(row["status"] == "PASS" for row in checks):
    failed = [row for row in checks if row["status"] != "PASS"]
    raise AssertionError(f"Sisyphus Watch demo checks failed: {failed}")

print("Demo mode PASS: card, branches, agent workflow trace, run summary, epistemic layers, version timeline, claim drift, claim graph, reviewer presets, evidence update simulation, revision comparison view, scenario authoring preview, revision packet, version diff, JSON, JSONL, and agent packet are available.")
print("Selected-card JSONL preview first 500 chars:")
print(to_jsonl(news_card)[:500] + "...")
print("All-demo-cards JSONL preview first 500 chars:")
print(to_all_cards_jsonl(all_news_cards)[:500] + "...")


## Kaggle Mid-Check Checklist

This final readout checks the reviewer path for the selected deterministic scenario: no API key, scenario load, trace, epistemic layer separation, human card, evidence update, revision comparison, evaluation, and `/kaggle/working` export configuration.


In [ ]:
display(HTML(render_kaggle_midcheck_summary_html(news_card, evidence_patch)))


## Limitations

- Sisyphus Watch is not an automatic truth oracle.
- It depends on source quality and source coverage.
- Strategic intent remains uncertain unless directly evidenced.
- Bias is labeled for review, not magically removed.
- Generated image prompts are not evidence.
- This notebook uses synthetic demo fixtures for safe, reproducible Kaggle evaluation.
- The demo does not implement live web ingestion, crawling, accounts, recommendations, a database, or a production news platform.

## Next Steps

1. Add optional JSON Schema validation with `jsonschema` when the dependency is available.
2. Package the notebook as a Kaggle Code submission with the included `data/`, `src/`, `schemas/`, and `examples/` folders.
3. Add a lightweight export command for producing JSONL from future scenarios.
